In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('df_final.csv')
df.head()

,ACCOUNT_ID,CARD_ID,TRANSACTION_ID,GROSS_TRANSACTION_AMOUNT,TRANSACTION_DATE,TRANSACTION_TYPE,TRANSACTION_STATE,TRANSACTION_CITY,MERCHANT_STATE,MERCHANT_CITY,...,CARD_HOLDER_VINTAGE,CARD_PRESENT_INDICATOR,MERCHANT_ID,MERCHANT_NAME,SHOPPER_CLASSIFICATION,TZ_NAME,LOCAL_TIME,Hour,DayOfWeek,CLUSTER_K4
0,A6184793,C7717436,T8535582,0.99,2024-03-13 06:46:55+00:00,Spend,OH,EUCLID,OH,WESTLAKE,...,29,Card Not Present,112,WENDYS,0,America/New_York,2024-03-13 02:46:55,2,Wednesday,1
1,A2296562,C5518346,T2383075,22.76,2024-05-16 20:34:18+00:00,Spend,MI,FORT GRATIOT,MI,FORT GRATIOT,...,87,Card Present,214,JIMMY JOHNS GOURMET SANDWICHES,2,America/New_York,2024-05-16 16:34:18,16,Thursday,0
2,A5523719,C7179772,T7494465,8.84,2024-04-25 17:09:37+00:00,Spend,OH,TOLEDO,OH,TOLEDO,...,25,Card Present,58,SUBWAY,0,America/New_York,2024-04-25 13:09:37,13,Thursday,1
3,A7905524,C1779514,T5897261,8.78,2024-05-26 13:50:06+00:00,Spend,IN,LAFAYETTE,IN,LAFAYETTE,...,26,Card Not Present,37,MCDONALDS,0,America/New_York,2024-05-26 09:50:06,9,Sunday,0
4,A7846687,C8240547,T2709494,18.01,2024-05-05 19:54:35+00:00,Spend,MN,DELANO,MN,DELANO,...,10,Card Present,522,CULVERS,0,America/Chicago,2024-05-05 14:54:35,14,Sunday,1


In [3]:
state_to_region = {
    # 동북부 (Northeast)
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast', 'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    # 중서부 (Midwest)
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest', 'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest', 'SD': 'Midwest',
    # 남부 (South)
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South', 'SC': 'South', 'VA': 'South', 'DC': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South', 'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South', 'TX': 'South',
    # 서부 (West)
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 'HI': 'West', 'OR': 'West', 'WA': 'West'
}

# 가맹점 위치(MERCHANT_STATE) 기준으로 REGION 칼럼 추가
df['REGION'] = df['CARD_HOLDER_STATE'].map(state_to_region).fillna('Unknown')

In [4]:
df.head()

,ACCOUNT_ID,CARD_ID,TRANSACTION_ID,GROSS_TRANSACTION_AMOUNT,TRANSACTION_DATE,TRANSACTION_TYPE,TRANSACTION_STATE,TRANSACTION_CITY,MERCHANT_STATE,MERCHANT_CITY,...,CARD_PRESENT_INDICATOR,MERCHANT_ID,MERCHANT_NAME,SHOPPER_CLASSIFICATION,TZ_NAME,LOCAL_TIME,Hour,DayOfWeek,CLUSTER_K4,REGION
0,A6184793,C7717436,T8535582,0.99,2024-03-13 06:46:55+00:00,Spend,OH,EUCLID,OH,WESTLAKE,...,Card Not Present,112,WENDYS,0,America/New_York,2024-03-13 02:46:55,2,Wednesday,1,Midwest
1,A2296562,C5518346,T2383075,22.76,2024-05-16 20:34:18+00:00,Spend,MI,FORT GRATIOT,MI,FORT GRATIOT,...,Card Present,214,JIMMY JOHNS GOURMET SANDWICHES,2,America/New_York,2024-05-16 16:34:18,16,Thursday,0,Midwest
2,A5523719,C7179772,T7494465,8.84,2024-04-25 17:09:37+00:00,Spend,OH,TOLEDO,OH,TOLEDO,...,Card Present,58,SUBWAY,0,America/New_York,2024-04-25 13:09:37,13,Thursday,1,Midwest
3,A7905524,C1779514,T5897261,8.78,2024-05-26 13:50:06+00:00,Spend,IN,LAFAYETTE,IN,LAFAYETTE,...,Card Not Present,37,MCDONALDS,0,America/New_York,2024-05-26 09:50:06,9,Sunday,0,Midwest
4,A7846687,C8240547,T2709494,18.01,2024-05-05 19:54:35+00:00,Spend,MN,DELANO,MN,DELANO,...,Card Present,522,CULVERS,0,America/Chicago,2024-05-05 14:54:35,14,Sunday,1,Midwest


In [5]:
import pandas as pd

def calculate_lift_and_rank(df, axis1_cols, axis2_col):
    """
    axis1_cols: 기준이 되는 그룹핑 컬럼 리스트 (예: ['REGION', 'CLUSTER_K4'])
    axis2_col: 분석 대상 컬럼 (예: 'MERCHANT_CATEGORY_LEVEL_3')
    """
    
    # 1. 전체 데이터 기준 Axis 2의 글로벌 비율 (기댓값)
    global_ratio = df[axis2_col].value_counts(normalize=True).rename('GLOBAL_RATIO')
    
    # 2. Axis 1(지역+군집) 기준으로 그룹핑하여 Count와 Amount(금액) 집계
    # 금액 컬럼명이 'GROSS_TRANSACTION_AMOUNT'라고 가정
    local_stats = df.groupby(axis1_cols + [axis2_col]).agg(
        COUNT=('TRANSACTION_ID', 'count'),
        TOTAL_AMOUNT=('GROSS_TRANSACTION_AMOUNT', 'sum')
    ).reset_index()
    
    # 3. Axis 1 그룹별 총 결제 건수 계산 (로컬 비율 계산용)
    group_totals = df.groupby(axis1_cols).size().rename('GROUP_TOTAL').reset_index()
    
    # 4. 데이터 병합 및 비율/Lift 계산
    result = local_stats.merge(group_totals, on=axis1_cols)
    result['LOCAL_RATIO'] = result['COUNT'] / result['GROUP_TOTAL']
    
    result = result.merge(global_ratio, left_on=axis2_col, right_index=True)
    result['LIFT'] = (result['LOCAL_RATIO'] / result['GLOBAL_RATIO']).round(3)
    
    # 5. 보기 좋게 컬럼 정리 (축1을 하나의 문자열로 결합)
    # 예: "South_3" 처럼 지역과 군집을 붙여서 보기 편하게 만듦
    result['AXIS_1'] = result[axis1_cols].astype(str).agg('_'.join, axis=1)
    result.rename(columns={axis2_col: 'AXIS_2'}, inplace=True)
    
    # 6. 그룹(AXIS_1) 내에서 Lift 기준으로 순위(Rank) 부여
    result['LIFT_RANK'] = result.groupby('AXIS_1')['LIFT'].rank(ascending=False, method='min').astype(int)
    
    # 최종 출력 컬럼 선택 및 정렬
    final_df = result[['AXIS_1', 'AXIS_2', 'LIFT', 'COUNT', 'TOTAL_AMOUNT', 'LIFT_RANK']]
    final_df = final_df.sort_values(by=['AXIS_1', 'LIFT_RANK'])
    
    return final_df

# ==========================================
# 실행 예시
# ==========================================
# 1. 지역+군집 별 '업종(MCL3)' Lift 분석
mcl3_lift_df = calculate_lift_and_rank(df, ['REGION', 'CLUSTER_K4'], 'MERCHANT_CATEGORY_LEVEL_3')
print("--- 지역&군집별 업종 Lift ---")
display(mcl3_lift_df.head(10))

# 2. 지역+군집 별 '가맹점(MERCHANT_NAME)' Lift 분석
merchant_lift_df = calculate_lift_and_rank(df, ['REGION', 'CLUSTER_K4'], 'MERCHANT_NAME')
print("\n--- 지역&군집별 가맹점 Lift ---")
display(merchant_lift_df.head(10))

--- 지역&군집별 업종 Lift ---


,AXIS_1,AXIS_2,LIFT,COUNT,TOTAL_AMOUNT,LIFT_RANK
29,Midwest_0,Mediterranean Restaurants,5.255,5,98.11,1
26,Midwest_0,Italian Restaurants,3.665,105,2247.28,2
11,Midwest_0,Casinos,3.274,3,9.91,3
5,Midwest_0,"Automobile Parts, Tires, and Services",2.183,1,2.65,4
1,Midwest_0,American Restaurants,1.835,12433,218454.42,5
22,Midwest_0,Grocery,1.792,1221,8023.27,6
24,Midwest_0,Health Care,1.577,1,8.78,7
40,Midwest_0,QSR Sandwiches,1.482,13640,247037.79,8
19,Midwest_0,Food Services,1.404,5040,66493.08,9
7,Midwest_0,Book Retailers,1.319,169,2009.64,10



--- 지역&군집별 가맹점 Lift ---


,AXIS_1,AXIS_2,LIFT,COUNT,TOTAL_AMOUNT,LIFT_RANK
33,Midwest_0,BRAUMS ICE CREAM & DAIRY STORES,29.641,1,2.70,1
58,Midwest_0,CITGO,29.641,1,11.88,1
90,Midwest_0,FACEBOOK,29.641,1,25.38,1
120,Midwest_0,HONEYBAKED HAM,29.641,1,29.94,1
227,Midwest_0,RUBY TUESDAY,29.641,1,73.98,1
121,Midwest_0,HORSESHOE,19.761,2,3.57,6
28,Midwest_0,BMO HARRIS BANK,14.821,3,42.68,7
173,Midwest_0,MO BETTAHS,14.821,1,31.93,7
193,Midwest_0,PANCHEROS,12.658,442,7875.27,9
69,Midwest_0,COUSINS SUBS,11.795,191,4076.46,10


In [6]:
import pandas as pd

def calculate_lift_and_rank(df, axis1_cols, axis2_cols):
    """
    axis1_cols: 기준이 되는 그룹핑 컬럼 리스트 (예: ['REGION', 'CLUSTER_K4'])
    axis2_cols: 타겟 변수 컬럼 리스트 (예: ['MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME'])
    """
    
    # 1. 전체 데이터 기준 Axis 2(업종+가맹점 조합)의 글로벌 비율 (기댓값)
    global_ratio = df.groupby(axis2_cols).size() / len(df)
    global_ratio = global_ratio.rename('GLOBAL_RATIO').reset_index()
    
    # 2. Axis 1(지역+군집) + Axis 2(업종+가맹점) 기준으로 그룹핑하여 Count와 Amount 집계
    local_stats = df.groupby(axis1_cols + axis2_cols).agg(
        COUNT=('TRANSACTION_ID', 'count'),
        TOTAL_AMOUNT=('GROSS_TRANSACTION_AMOUNT', 'sum')
    ).reset_index()
    
    # 3. Axis 1 그룹별 총 결제 건수 계산 (로컬 비율 계산용)
    group_totals = df.groupby(axis1_cols).size().rename('GROUP_TOTAL').reset_index()
    
    # 4. 데이터 병합 및 비율/Lift 계산
    result = local_stats.merge(group_totals, on=axis1_cols)
    result['LOCAL_RATIO'] = result['COUNT'] / result['GROUP_TOTAL']
    
    result = result.merge(global_ratio, on=axis2_cols)
    result['LIFT'] = (result['LOCAL_RATIO'] / result['GLOBAL_RATIO']).round(3)
    
    # 5. 보기 좋게 컬럼 정리 (축을 하나의 문자열로 결합)
    # AXIS_1 예: "South_3"
    result['AXIS_1'] = result[axis1_cols].astype(str).agg('_'.join, axis=1)
    
    # AXIS_2 예: "QSR Burgers - WENDYS" (업종 먼저, 그 다음 가맹점)
    result['AXIS_2'] = result[axis2_cols].astype(str).agg(' - '.join, axis=1)
    
    # 6. 그룹(AXIS_1) 내에서 Lift 기준으로 순위(Rank) 부여
    result['LIFT_RANK'] = result.groupby('AXIS_1')['LIFT'].rank(ascending=False, method='min').astype(int)
    
    # 7. 최종 출력 컬럼 선택 및 정렬
    final_df = result[['AXIS_1', 'AXIS_2', 'LIFT', 'COUNT', 'TOTAL_AMOUNT', 'LIFT_RANK']]
    final_df = final_df.sort_values(by=['AXIS_1', 'LIFT_RANK'])
    
    return final_df

# ==========================================
# 실행 예시
# ==========================================
# Axis 1: 지역 + 군집
# Axis 2: 업종 -> 가맹점명 순서로 결합
combined_lift_df = calculate_lift_and_rank(
    df, 
    axis1_cols=['REGION', 'CLUSTER_K4'], 
    axis2_cols=['MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME']
)

print("--- [REGION_CLUSTER] vs [업종 - 가맹점명] Lift 분석 ---")
display(combined_lift_df)

--- [REGION_CLUSTER] vs [업종 - 가맹점명] Lift 분석 ---


,AXIS_1,AXIS_2,LIFT,COUNT,TOTAL_AMOUNT,LIFT_RANK
34,Midwest_0,American Restaurants - RUBY TUESDAY,29.641,1,73.98,1
96,Midwest_0,Dessert - BRAUMS ICE CREAM & DAIRY STORES,29.641,1,2.70,1
128,Midwest_0,Gas / Convenience - CITGO,29.641,1,11.88,1
133,Midwest_0,General Retail - HONEYBAKED HAM,29.641,1,29.94,1
71,Midwest_0,Casinos - HORSESHOE,19.761,2,3.57,5
...,...,...,...,...,...,...
6011,West_3,Mexican Restaurants - PANCHEROS,0.023,1,9.95,345
6088,West_3,QSR Chicken - BOJANGLES FAMOUS CHICKEN N BISCUITS,0.017,22,506.04,346
5943,West_3,Dessert - SWEETFROG,0.016,1,4.87,347
5833,West_3,American Restaurants - JACKS FAMILY RESTAURANTS,0.009,3,30.21,348


In [7]:
combined_lift_df[combined_lift_df['COUNT']>10000]

,AXIS_1,AXIS_2,LIFT,COUNT,TOTAL_AMOUNT,LIFT_RANK
86,Midwest_0,Coffee / Tea - STARBUCKS,1.123,10815,155177.58,108
222,Midwest_0,QSR Burgers - MCDONALDS,1.057,26120,303910.53,114
306,Midwest_1,American Restaurants - CULVERS,1.953,10981,182147.41,73
307,Midwest_1,American Restaurants - DAIRY QUEEN,1.600,11870,159233.54,99
649,Midwest_1,Vending & Beverage Retailers - 365 RETAIL MARKETS,1.417,31153,126242.82,108
...,...,...,...,...,...,...
5553,West_2,Coffee / Tea - STARBUCKS,1.535,22587,350473.52,166
5729,West_2,QSR Burgers - MCDONALDS,0.897,33887,432734.25,230
5901,West_3,Coffee / Tea - NO ENTITY,1.739,10232,130141.37,148
5907,West_3,Coffee / Tea - STARBUCKS,1.468,17832,266622.35,158
